In [11]:
import os,sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.append('/root/liubo/TravDiT')  # 添加项目根目录到 Python 路径
from args import make_args
from util import load_raw_data,load_vec,generate_trajectory_prompt_temp,temporal_pattern_one_hot_batch
import random
from tqdm import tqdm 
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args = make_args()
unique_poi_types = args.unique_poi_types
random.seed(args.seed)
task = args.task
print(task)

In [12]:
# 预训练Transformer encoder for DiT
from model.model_Encoder import ST_Encoder,ST_Decoder
from model.st_layers_config.args import parse_args
encoder_config = parse_args()
encoder = ST_Encoder(config = encoder_config,dim_in = args.poi_dim+args.pos_dim, dim_out = args.latent_dim)
encoder.load_state_dict(torch.load(f"/root/liubo/TravDiT/{args.checkpt_path}/tuning/encoder/encoder_2to1_gz_few.pth", map_location=device))
encoder.to(device).eval()

from model.model_Projector import Projector
cityproj = Projector(latent_dim=args.latent_dim, vocab_size=args.vocab_size)
cityproj.to(device)

Projector(
  (aligner): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=128, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): GELU(approximate='none')
    (5): Linear(in_features=128, out_features=128, bias=True)
  )
  (decoder): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=128, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): GELU(approximate='none')
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=890, bias=True)
  )
)

In [13]:
from dataset.custom_dataset import CustomDataset
from torch.utils.data import DataLoader

all_data = []
dow_map = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
city_list = [args.city]
sample_size = 5000 # for one city

for city_id, city in enumerate(city_list):
    raw_data = load_raw_data(city=city)
    vec = load_vec(city=city).to(device)
    vec = vec[:, 2:2+args.poi_dim+args.pos_dim]

    indices = random.sample(range(len(raw_data)), sample_size)

    for i in indices:
        item = raw_data[i]
        item['city_id'] = city_id   # ✅ 添加 city_id 字段
        vec_seq = torch.stack([vec[int(rid)] for rid in item['traj_region_id']])
        all_data.append((item, vec_seq))

# 原始数据先传入 CustomDataset
random.shuffle(all_data)
all_raw_data, all_vec_seq = zip(*all_data)
all_raw_data = list(all_raw_data)
all_vec_seq = list(all_vec_seq)

full_dataset = CustomDataset(all_raw_data, all_vec_seq)
# 再拆分 Subset
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
# 用 DataLoader 加载 Subset
train_dataloader = DataLoader(train_dataset, batch_size=args.Encoder_batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=args.Encoder_batch_size, shuffle=False)

[INFO] Generated spatial POI+pos vec: torch.Size([890, 20])


In [14]:
from torch.optim import AdamW

optimizer = AdamW(list(cityproj.parameters()), lr=1e-2, weight_decay=0.01)
# optimizer = AdamW([
#     {"params": cityproj.parameters(), "lr": 1e-2},        # 对 cityproj 使用较大学习率
#     {"params": encoder.parameters(), "lr": 1e-3},         # 对 encoder 使用较小学习率
# ], weight_decay=0.01)
total_steps = len(train_dataloader)*args.alignment_epoch
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup

from transformers import get_linear_schedule_with_warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = torch.nn.CrossEntropyLoss()  # ignore non-masked labels，这里看的是label是否mask


In [15]:
from sklearn.metrics import accuracy_score

for epoch in range(args.alignment_epoch):
    epoch_loss = 0
    dataloader_tqdm = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{args.alignment_epoch}", leave=False)
    cityproj.train()
    for batch in dataloader_tqdm:
        region_id = batch['region_seq'].to(device)  # [B, K]
        labels = batch['region_seq'].to(device)        # [B, K]
        city_id = batch['city_id_seq'].to(device)      # [B, ]
        vec = batch['vec_seq'].to(device)  # [B, K, poi_dim + pos_dim]

        labels = labels.reshape(-1, args.K).long()  # [B*K]
        city_id = city_id.unsqueeze(1).expand(-1, args.K)  # [B, K]

        latent = encoder(input=vec.permute(0, 2, 1).unsqueeze(2))  # -> [B, T, 1, C] 
        aligned_output = cityproj(latent)                                # -> [B, T, vocab_size]

        loss = criterion(aligned_output.view(-1, aligned_output.size(-1)), labels.view(-1))
        loss.backward()
        scheduler.step()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        dataloader_tqdm.set_postfix(loss=loss.item())
        avg_loss = epoch_loss / len(train_dataloader)
    print(f"[Epoch {epoch+1}/{args.alignment_epoch}] Average Loss: {avg_loss:.6f}")

    with torch.no_grad():
        cityproj.eval()
        epoch_loss = 0
        all_preds = []
        all_trues = []

        for batch in val_dataloader:
            region_id = batch['region_seq'].to(device)  # [B, K]
            labels = batch['region_seq'].to(device)        # [B, K]
            city_id = batch['city_id_seq'].to(device)      # [B, ]
            vec = batch['vec_seq'].to(device)  # [B, K, poi_dim + pos_dim]

            labels = labels.reshape(-1, args.K).long()  # [B*K]
            city_id = city_id.unsqueeze(1).expand(-1, args.K)  # [B, K]

            latent = encoder(input=vec.permute(0, 2, 1).unsqueeze(2))  # -> [B, T, 1, C]
            aligned_output = cityproj(latent)                                # -> [B, T, vocab_size]

            loss = criterion(aligned_output.view(-1, aligned_output.size(-1)), labels.view(-1))
            epoch_loss += loss.item()

            # 收集预测和标签
            y_true = labels.view(-1).cpu()
            y_pred = aligned_output.argmax(dim=-1).view(-1).cpu()
            all_trues.append(y_true)
            all_preds.append(y_pred)

        # 拼接所有预测和标签
        all_trues = torch.cat(all_trues)
        all_preds = torch.cat(all_preds)
        val_acc = accuracy_score(all_trues.numpy(), all_preds.numpy())

        print(f"[Epoch {epoch+1}] Val Loss: {epoch_loss/len(val_dataloader):.4f} | Val Accuracy: {val_acc:.4f}")


Epoch 1/50:   0%|          | 0/16 [00:00<?, ?it/s]/root/liubo/TravDiT/.conda/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[Epoch 1/50] Average Loss: 6.743548
[Epoch 1] Val Loss: 6.5211 | Val Accuracy: 0.0058


[Epoch 2/50] Average Loss: 6.061093
[Epoch 2] Val Loss: 5.5344 | Val Accuracy: 0.0247


[Epoch 3/50] Average Loss: 4.993977
[Epoch 3] Val Loss: 4.4661 | Val Accuracy: 0.1048


[Epoch 4/50] Average Loss: 4.110903
[Epoch 4] Val Loss: 3.8573 | Val Accuracy: 0.1523


...

[Epoch 46/50] Average Loss: 0.578562
[Epoch 46] Val Loss: 4.3830 | Val Accuracy: 0.3406


[Epoch 47/50] Average Loss: 0.571321
[Epoch 47] Val Loss: 4.3980 | Val Accuracy: 0.3403


[Epoch 48/50] Average Loss: 0.569699
[Epoch 48] Val Loss: 4.4203 | Val Accuracy: 0.3385


[Epoch 49/50] Average Loss: 0.563195
[Epoch 49] Val Loss: 4.4236 | Val Accuracy: 0.3391


[Epoch 50/50] Average Loss: 0.560502
[Epoch 50] Val Loss: 4.4248 | Val Accuracy: 0.3385


In [16]:
# 保存模型 
torch.save(cityproj.state_dict(), f"/root/liubo/TravDiT/{args.checkpt_path}/tuning/projector/projector_{task}_2to1_gz_few_5k.pth")